# Topic 1: Multi-Decoder Uncertainty-Triggered RAG (Kaggle GPU)
**Author:** Khushi Shukla  
**Model:** `allenai/OLMo-2-1124-7B-Instruct`  
**Protocol:** See `PROTOCOL.md` in repo

## Before running:
1. Enable **GPU accelerator** (T4 x2 or P100) in notebook settings → right panel
2. Enable **Internet** in notebook settings (required for HuggingFace)
3. Add `HF_TOKEN` to Kaggle Secrets: Account → Settings → Secrets
4. Run Cell 1 (setup) → Cell 2 (dev only) → review tau CIs → Cell 3 (test)

In [ ]:
# ── CELL 1: Environment setup ─────────────────────────────────────────────────
import os

# 1a. Optional HF_TOKEN (OLMo-2 and TriviaQA are public, so token is optional)
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ['HF_TOKEN'] = secrets.get_secret('HF_TOKEN')
    print('HF_TOKEN loaded from Kaggle Secrets ✓')
except Exception:
    print('No Kaggle Secret found. OLMo-2 is open access, proceeding without token ✓')

# 1b. Install dependencies
!pip install -q transformers datasets accelerate bitsandbytes scikit-learn scipy rank_bm25

# 1c. Clone or update repo
REPO_DIR = '/kaggle/working/token_retrieval'
if os.path.exists(REPO_DIR):
    print('Repo already exists, pulling latest...')
    !git -C {REPO_DIR} pull origin main
else:
    !git clone https://github.com/Khushi281300/token_retrieval.git {REPO_DIR}

os.chdir(REPO_DIR)
!pwd && git log --oneline -3

# 1d. GPU check
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

In [ ]:
# ── CELL 2: PHASE 1 — Dev calibration ONLY (500 questions) ───────────────────
# Runs: threshold calibration at 20%/40%/60%, bootstrap CIs, Cohen's kappa,
# teacher-forced control, baselines on dev.
# Does NOT touch the test set.
#
# Expected runtime: ~90-120 min on T4 (with 4-bit loading)
# After this cell, check the tau CIs, then update PROTOCOL.md equivalence margins
# before running Cell 3.

!python scripts/run_experiments.py \
    --model allenai/OLMo-2-1124-7B-Instruct \
    --dataset trivia_qa \
    --num_dev 500 \
    --num_test 1000 \
    --max_draft_tokens 64 \
    --max_answer_tokens 32 \
    --score_type raw \
    --unit nats \
    --load_in_4bit \
    --run_dev_only \
    --output_dir runs

In [ ]:
# ── CELL 2b: Review dev tau calibration & baseline metrics ───────────────────
# Run this after Cell 2 finishes to inspect all dev metrics before touching test.
import json, glob, os

# Search across both default runs/ and custom output dirs
search_patterns = [
    'runs/dev_calibration_summary_*.json',
    '/kaggle/working/runs_dev500/dev_calibration_summary_*.json',
    'runs/full_experiment_*.json',
    '/kaggle/working/runs_dev500/full_experiment_*.json'
]
run_files = []
for p in search_patterns:
    run_files.extend(glob.glob(p))
run_files = sorted(set(run_files))

if not run_files:
    print('No dev summary files found yet. Run Cell 2 first.')
else:
    latest = run_files[-1]
    with open(latest) as f:
        data = json.load(f)

    meta = data.get('metadata', {})
    print(f'Latest summary file: {latest}')
    print(f'Git commit hash:     {meta.get("git_commit_hash", "unknown")}')
    print(f'Dev questions (N):   {meta.get("num_dev_questions", 500)}')
    print(f'Pooled corpus size:  {meta.get("num_corpus_passages", 2600)}')
    print(f'Compute precision:   {meta.get("compute_dtype", "torch.float16")}')

    if 'dev_calibration' in data:
        print('\n=== 1. DEV TAU CALIBRATION (with 95% Bootstrap CIs) ===')
        for dec, rates in data['dev_calibration'].items():
            print(f'  {dec:<20}:')
            for rate, ci in rates.items():
                print(f'    Rate {rate}: tau = {ci["tau"]:.4f} [95% CI: {ci["ci_lower"]:.4f}, {ci["ci_upper"]:.4f}]')

    if 'dev_cohen_kappa_vs_greedy' in data:
        print('\n=== 2. DEV TRIGGER AGREEMENT (Cohen\'s Kappa vs Greedy @ 40%) ===')
        for dec, kappa in data['dev_cohen_kappa_vs_greedy'].items():
            print(f'  {dec:<20}: kappa = {kappa:.4f}')

    if 'dev_teacher_forced_control' in data:
        print('\n=== 3. TEACHER-FORCED CONTROL (Scoring on Identical Greedy Drafts) ===')
        for dec, stats in data['dev_teacher_forced_control'].items():
            print(f'  {dec:<20}: Mean Max-Entropy = {stats["mean_max_entropy"]:.4f}, '
                  f'Trigger Rate under Greedy tau = {stats["retrieval_rate_under_greedy_tau"]:.1%}')

    if 'dev_baselines' in data:
        b = data['dev_baselines']
        print('\n=== 4. DEV BASELINE & CORPUS METRICS ===')
        print(f'  No-Retrieval:     F1 = {b.get("no_retrieval",{}).get("f1",0):.4f} | EM = {b.get("no_retrieval",{}).get("em",0):.4f}')
        print(f'  Always-Retrieve:  F1 = {b.get("always_retrieve",{}).get("f1",0):.4f} | EM = {b.get("always_retrieve",{}).get("em",0):.4f} | Recall@3 = {b.get("always_retrieve",{}).get("recall_at_3",0):.1%}')
        print(f'  Oracle:           F1 = {b.get("oracle",{}).get("f1",0):.4f} | EM = {b.get("oracle",{}).get("em",0):.4f}')
        delta = b.get('retrieval_delta_f1', {})
        print(f'  Retrieval Delta:  ΔF1 = {delta.get("mean",0):+.4f} [95% CI: {delta.get("ci_95",[0,0])[0]:+.4f}, {delta.get("ci_95",[0,0])[1]:+.4f}, SE = {delta.get("se_f1",0):.4f}]')

    if 'dev_decoder_stats' in data:
        print('\n=== 5. DEV DRAFT GENERATION METRICS (Length & Repetition) ===')
        for dec, st in data['dev_decoder_stats'].items():
            print(f'  {dec:<20}: Mean Length = {st["mean_draft_len"]} tok | Distinct-1 = {st["distinct_1"]} | Distinct-2 = {st["distinct_2"]} | Distinct-3 = {st["distinct_3"]}')

    print('\n>>> ACTION: All dev numbers documented! Update PROTOCOL.md margins before running Cell 3 (Test).')


In [ ]:
# ── CELL 3: PHASE 3+4 — Full test run (1,000 questions) ───────────────────────
# !! ONLY RUN THIS AFTER:  !!
#   1. Cell 2 (dev calibration) has finished
#   2. You have reviewed the tau CIs in Cell 2b
#   3. You have written equivalence margins in PROTOCOL.md
#
# Uses --skip_dev to load the frozen dev thresholds from configs/dev_calibrated_taus.json,
# so it immediately starts evaluating the 1,000 test questions without repeating the dev run!
#
# Expected runtime: ~2.5-3 hours on T4

assert 'MARGINS FROZEN:' in open('PROTOCOL.md').read(), 'Freeze margins in PROTOCOL.md first!'

!python scripts/run_experiments.py \
    --model allenai/OLMo-2-1124-7B-Instruct \
    --dataset trivia_qa \
    --num_dev 500 \
    --num_test 1000 \
    --max_draft_tokens 64 \
    --max_answer_tokens 32 \
    --score_type raw \
    --unit nats \
    --load_in_4bit \
    --skip_dev \
    --output_dir runs

In [ ]:
# ── CELL 4: Save results to Kaggle output & print final tables ────────────────
import json, glob, shutil

run_files = sorted(glob.glob('runs/full_experiment_*.json'))
jsonl_files = sorted(glob.glob('runs/experiment_records_*.jsonl'))

if run_files:
    latest_json = run_files[-1]
    latest_jsonl = jsonl_files[-1] if jsonl_files else None

    # Copy to /kaggle/working so they appear in notebook output panel
    shutil.copy(latest_json, f'/kaggle/working/{os.path.basename(latest_json)}')
    if latest_jsonl:
        shutil.copy(latest_jsonl, f'/kaggle/working/{os.path.basename(latest_jsonl)}')

    with open(latest_json) as f:
        data = json.load(f)

    print(f'Results saved to Kaggle output: {os.path.basename(latest_json)}')
    print(f'Git hash: {data["metadata"]["git_commit_hash"]}')
    print(f'Dev N: {data["metadata"]["num_dev_questions"]} | Test N: {data["metadata"]["num_test_questions"]}')
    print()

    # TABLE 1
    tau_fixed = list(data['dev_calibration']['greedy'].values())[1]['tau']  # 40% rate
    print(f'=== TABLE 1: FIXED-THRESHOLD COMPARISON (tau_fixed={tau_fixed:.4f}) ===')
    print(f'{"Draft Decoder":<22} | {"Rate":<8} | {"F1":<8} | {"EM"}')
    print('-'*55)
    for dec, s in data['table1_fixed_threshold'].items():
        std_r = f" ±{s['retrieval_rate_std']:.1%}" if s['retrieval_rate_std'] > 0 else ''
        std_f = f" ±{s['f1_std']:.4f}" if s['f1_std'] > 0 else ''
        print(f"{dec:<22} | {s['retrieval_rate']:.1%}{std_r:<6} | {s['f1']:.4f}{std_f:<8} | {s['em']:.4f}")

    print()
    print('=== TABLE 2: MATCHED-RATE COMPARISON (tau tuned per decoder @ 40%) ===')
    print(f'{"Draft Decoder":<22} | {"Tau (95% CI)":<18} | {"Rate":<8} | {"F1":<8} | {"EM"}')
    print('-'*72)
    for dec, s in data['table2_matched_rate'].items():
        ci = s['tau_ci']
        std_f = f" ±{s['f1_std']:.4f}" if s['f1_std'] > 0 else ''
        print(f"{dec:<22} | {s['tau_matched']:.4f} [{ci[0]:.2f},{ci[1]:.2f}]  | {s['retrieval_rate']:.1%}      | {s['f1']:.4f}{std_f:<8} | {s['em']:.4f}")

    print()
    print('=== TABLE 3: ANSWER DECODER CONTEXT COPY PENALTY (RQ1c) ===')
    print(f'{"Answer Decoder":<22} | {"F1":<8} | {"EM":<8} | {"Copy Rate 3-gram"}')
    print('-'*58)
    for dec, s in data['table3_answer_decoder'].items():
        print(f"{dec:<22} | {s['f1']:.4f}   | {s['em']:.4f}   | {s['copy_rate_3']:.4f}")

    print()
    print('=== TABLE 4: BASELINES ===')
    for b, s in data['table4_baselines'].items():
        print(f"  {b:<25} F1={s['f1']:.4f}  EM={s['em']:.4f}  Rate={s.get('retrieval_rate',0):.1%}")
else:
    print('No results yet — run Cell 3 first.')